# 🚀 ZAC 2026 LifeMate — Version 8.0 (`AIO_HacDaiBangBang`)

### 📋 Cấu trúc 8 Ô dễ quản lý & hiển thị trực tiếp tiến độ từng bước (`tqdm` + Epoch/ETA):
- **Ô 1:** Kết nối Google Drive · Đồng bộ code `v8.0` từ GitHub · Cài đặt thư viện & Kiểm tra GPU.
  - **Ô 2:** Kiểm tra Unit Tests Guardrails `v8.0` & Tạo Train/Val Splits.
  - **Ô 3:** Dịch bù 92 file âm thanh lịch sử cho 42 mẫu Hội thoại dài (`18–24` lượt, `max_history_user_audios=30`) & Chuẩn hóa email đọc bằng lời.
  - **Ô 4:** Xây dựng tập dữ liệu `sft_train_v8.jsonl` (`980` mẫu tinh gọn = **1.0 Epoch / 122 bước**).
  - **Ô 5:** Warm-Start huấn luyện `qwen3_4b_sft_lora_v8` từ `v7` (**1.0 Epoch = 122 bước**, in tiến độ Epoch + thời gian còn lại mỗi 5 bước, tự động lưu checkpoint mỗi **20 bước** lên Drive).
  - **Ô 6:** Suy luận `public_test` `v8.0` $\rightarrow$ Ghi trực tiếp `/content/drive/MyDrive/ZaloAI2026_outputs_v8/submission_v8.jsonl` (đóng file sau mỗi mẫu để đồng bộ Drive tức thì).
  - **Ô 7:** Kiểm tra hợp lệ Schema `zaic_tools` · Thống kê Timing & So sánh `v5` / `v6` / `v7` / `v8`.
  - **Ô 8 (Đóng gói `saved_models/` nộp Docker cho đội `AIO_HacDaiBangBang`):** Hợp nhất (`merge`) `qwen3_4b_sft_lora_v8` vào base model với `max_position_embeddings = 16384` để tạo thư mục `saved_models/`.


In [ ]:
# === Ô 1: Kết nối Google Drive · Đồng bộ Code v8.0 từ GitHub · Cài đặt Thư viện & Kiểm tra GPU ===
import os, sys, glob, shutil, zipfile, subprocess, time, json
from pathlib import Path

# 0) Dọn sạch tiến trình ngầm cũ (nếu có) để giải phóng 100% VRAM
subprocess.run(["pkill", "-9", "-f", "scripts/infer.py"], check=False)
subprocess.run(["pkill", "-9", "-f", "scripts/train_sft_lora.py"], check=False)
time.sleep(2)

# 1) Kết nối Google Drive & Đồng bộ code v8.0 mới nhất từ GitHub
LOCAL_DIR = Path("/home/leatuyrbertyk/Documents/ZAIC26")
if LOCAL_DIR.is_dir():
    SRC_DIR  = str(LOCAL_DIR)
    WORK_DIR = f"{SRC_DIR}/lifemate_code"
    OUT_DIR  = SRC_DIR
    V7_DIR   = SRC_DIR
else:
    if not os.path.isdir("/content/drive/MyDrive"):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(
                "❌ Lỗi kết nối Google Drive: VS Code Colab Extension không hỗ trợ hộp thoại cấp quyền drive.mount(). "
                "Bạn hãy mở và chạy notebook này trực tiếp trên trình duyệt web https://colab.research.google.com nhé!"
            ) from e
    candidates = (
        glob.glob("/content/drive/MyDrive/ZaloAI2026")
        + glob.glob("/content/drive/.shortcut-targets-by-id/*/ZaloAI2026")
        + glob.glob("/content/drive/Shareddrives/*/ZaloAI2026")
        + glob.glob("/content/drive/Shareddrives/ZaloAI2026")
    )
    if not candidates:
        raise FileNotFoundError("❌ Không tìm thấy thư mục ZaloAI2026 trên Google Drive!")
    SRC_DIR  = candidates[0]
    WORK_DIR = "/content/lifemate_project"
    V7_DIR   = "/content/drive/MyDrive/ZaloAI2026_outputs_v7"
    OUT_DIR  = "/content/drive/MyDrive/ZaloAI2026_outputs_v8"
    os.makedirs(WORK_DIR, exist_ok=True)
    os.makedirs(OUT_DIR, exist_ok=True)

    if not os.path.isdir("/content/zaic26_git/.git"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/LeatuyrBertyk/ZAIC26.git", "/content/zaic26_git"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/zaic26_git", "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", "/content/zaic26_git", "reset", "--hard", "origin/main"], check=True)
    if os.path.isdir("/content/zaic26_git/lifemate_code"):
        shutil.copytree("/content/zaic26_git/lifemate_code", WORK_DIR, dirs_exist_ok=True)
    elif os.path.isdir(f"{SRC_DIR}/lifemate_code"):
        shutil.copytree(f"{SRC_DIR}/lifemate_code", WORK_DIR, dirs_exist_ok=True)

# 2) Chuẩn bị DATA_ROOT
if os.path.isfile(f"{SRC_DIR}/train+public_test/train/train.jsonl"):
    DATA_ROOT = f"{SRC_DIR}/train+public_test"
elif os.path.isfile(f"{SRC_DIR}/train+public_test/train+public_test/train/train.jsonl"):
    DATA_ROOT = f"{SRC_DIR}/train+public_test/train+public_test"
else:
    RAW_DIR = "/content/data_raw"
    if not glob.glob(f"{RAW_DIR}/**/public_test/public_test.jsonl", recursive=True):
        os.makedirs(RAW_DIR, exist_ok=True)
        with zipfile.ZipFile(f"{SRC_DIR}/train+public_test.zip") as z:
            z.extractall(RAW_DIR)
    hit = glob.glob(f"{RAW_DIR}/**/public_test/public_test.jsonl", recursive=True)[0]
    DATA_ROOT = os.path.dirname(os.path.dirname(hit))

link_parent = f"{WORK_DIR}/train+public_test"
link_target = f"{link_parent}/train+public_test"
os.makedirs(link_parent, exist_ok=True)
if not os.path.lexists(link_target):
    os.symlink(DATA_ROOT, link_target)

def get_path(name: str) -> str:
    for d in (OUT_DIR, V7_DIR, SRC_DIR):
        p = f"{d}/{name}"
        if os.path.exists(p):
            return p
    return f"{OUT_DIR}/{name}"

# 3) Kiểm tra GPU & Cài đặt thư viện
ret = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
GPU_NAME = ret.stdout.strip() if ret.returncode == 0 else "CPU"
SMALL_GPU = any(k in GPU_NAME for k in ("L4", "T4", "CPU"))
TIME_BUDGET = 150 if SMALL_GPU else 25

!pip install -q faster-whisper "transformers>=4.51.0" accelerate jiwer soundfile librosa
import torch
DTYPE     = "bfloat16" if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else "float16"
PHO_OUT   = get_path("asr_pho_ft_v1")
TURBO_OUT = get_path("asr_turbo_ft_v1")
os.chdir(WORK_DIR)

print(f"✅ GPU       = {GPU_NAME} (dtype={DTYPE} | time-budget={TIME_BUDGET}s)")
print(f"📂 SRC_DIR   = {SRC_DIR}")
print(f"📂 WORK_DIR  = {WORK_DIR}")
print(f"📂 DATA_ROOT = {DATA_ROOT}")
print(f"📂 OUT_DIR   = {OUT_DIR}")


In [ ]:
# === Ô 2: Kiểm tra Unit Tests Guardrails v8.0 & Tạo Train/Val Splits ===
import os
os.chdir(WORK_DIR)
if not os.path.isfile("outputs/splits/val_split.jsonl"):
    !python3 -X utf8 -u scripts/make_splits.py
!python3 -X utf8 -u scripts/test_agent_guardrails.py
!python3 -X utf8 -u scripts/oracle_agent_check.py --input outputs/splits/val_split.jsonl --show 3


In [ ]:
# === Ô 3: Dịch bù 92 file âm thanh lịch sử cho 42 mẫu Long-Context (18-24 lượt) & Chuẩn hóa Email ===
import os, shutil, time
os.chdir(WORK_DIR)
ASR_CACHE_FT = f"{OUT_DIR}/public_test_asr_ft_v1_cache.json"
if not os.path.isfile(ASR_CACHE_FT):
    src_cache = get_path("public_test_asr_ft_v1_cache.json")
    if os.path.isfile(src_cache):
        shutil.copy2(src_cache, ASR_CACHE_FT)

t0 = time.perf_counter()
!python3 -X utf8 -u scripts/infer.py --data-root "{DATA_ROOT}" --split public_test \
  --asr-mode transformers --asr-model "{TURBO_OUT}" --asr-secondary "{PHO_OUT}" \
  --asr-cache "{ASR_CACHE_FT}" --asr-only
print(f"✅ Hoàn tất kiểm tra & dịch bù ASR public_test trong {time.perf_counter()-t0:.1f}s -> {ASR_CACHE_FT}")


In [ ]:
# === Ô 4: Xây dựng tập dữ liệu SFT v8.0 tinh gọn (980 mẫu = 1.0 Epoch / 122 bước Warm-Start) ===
import os
os.chdir(WORK_DIR)
SFT_V8_PATH = f"{OUT_DIR}/sft_train_v8.jsonl"
if os.path.isfile(SFT_V8_PATH):
    n_lines = sum(1 for _ in open(SFT_V8_PATH, encoding="utf-8"))
    print(f"✅ Đã có sẵn tập dữ liệu {SFT_V8_PATH} ({n_lines} mẫu).")
else:
    val_cache = get_path("asr_val_ft_cache.json")
    asr_cache_arg = f'--asr-cache "{val_cache}"' if os.path.isfile(val_cache) else ''
    !python3 -X utf8 -u scripts/build_sft_data.py \
      --input-jsonl "{DATA_ROOT}/train/train.jsonl" \
      --v8-focused --synth-repeat 3 --recovery-count 100 \
      --include-source-b --source-b-count 240 {asr_cache_arg} \
      --out "{SFT_V8_PATH}"
    print(f"✅ Đã tạo xong tập dữ liệu SFT v8.0 tại: {SFT_V8_PATH}")


In [ ]:
# === Ô 5: Huấn luyện Warm-Start Qwen3-4B PureLoRA v8.0 từ v7 (1.0 Epoch = 122 bước, in tiến độ mỗi 5 bước, lưu mỗi 20 bước) ===
# Sử dụng !python3 -u để hiển thị trực tiếp thanh tiến trình tqdm (1/122) và thời gian còn lại (ETA) theo thời gian thực!
import os, shutil, subprocess
if os.path.isdir("/content/zaic26_git/.git"):
    subprocess.run(["git", "-C", "/content/zaic26_git", "fetch", "--depth", "1", "origin", "main"], check=False)
    subprocess.run(["git", "-C", "/content/zaic26_git", "reset", "--hard", "origin/main"], check=False)
    shutil.copytree("/content/zaic26_git/lifemate_code", WORK_DIR, dirs_exist_ok=True)
os.chdir(WORK_DIR)
SFT_V8_PATH = f"{OUT_DIR}/sft_train_v8.jsonl"
LORA_V8_OUT = f"{OUT_DIR}/qwen3_4b_sft_lora_v8"
LORA_V7_DIR = get_path("qwen3_4b_sft_lora_v7")
LORA_V5_DIR = get_path("qwen3_4b_sft_lora_v5")

if os.path.isfile(f"{LORA_V8_OUT}/epoch_1/pure_lora_weights.pt"):
    print(f"✅ Đã hoàn tất huấn luyện Warm-Start v8.0 (1.0 Epoch) tại: {LORA_V8_OUT}")
else:
    if os.path.isfile(f"{LORA_V8_OUT}/pure_lora_weights.pt"):
        init_adapter = LORA_V8_OUT
    elif os.path.isfile(f"{LORA_V7_DIR}/pure_lora_weights.pt"):
        init_adapter = LORA_V7_DIR
    else:
        init_adapter = LORA_V5_DIR
    resume_arg = f'--resume-from-adapter "{init_adapter}"' if os.path.isfile(f"{init_adapter}/pure_lora_weights.pt") else ''
    print(f"🔄 Bắt đầu Warm-Start Qwen3-4B PureLoRA v8.0 (Chỉ chạy 1.0 Epoch = 122 bước, ~50s/bước trên L4) khởi tạo từ: {init_adapter}")
    !python3 -X utf8 -u scripts/train_sft_lora.py \
      --base-model Qwen/Qwen3-4B-Instruct-2507 \
      --train-jsonl "{SFT_V8_PATH}" \
      --output-dir "{LORA_V8_OUT}" \
      --epochs 1.0 --lr 4e-5 --batch-size 1 --grad-accum 8 \
      --lora-r 32 --lora-alpha 64 --free-text-weight 0.3 --save-steps 20 {resume_arg}


In [ ]:
# === Ô 6: Suy luận public_test v8.0 -> Xuất submission_v8.jsonl (Đồng bộ Drive sau mỗi mẫu) ===
import os
os.chdir(WORK_DIR)
ASR_CACHE_FT   = f"{OUT_DIR}/public_test_asr_ft_v1_cache.json"
BASE_ASR_CACHE = get_path("public_test_whisper_turbo_cache.json")
LORA_V8_OUT    = f"{OUT_DIR}/qwen3_4b_sft_lora_v8"
LORA_V7_DIR    = get_path("qwen3_4b_sft_lora_v7")
PRIMARY_LORA   = LORA_V8_OUT if os.path.isfile(f"{LORA_V8_OUT}/pure_lora_weights.pt") else LORA_V7_DIR
SUBMISSION_V8  = f"{OUT_DIR}/submission_v8.jsonl"
TIMING_V8      = f"{OUT_DIR}/timing_public_v8.jsonl"
base_cache_arg = f'--asr-base-cache "{BASE_ASR_CACHE}"' if os.path.isfile(BASE_ASR_CACHE) else ''

print(f"🚀 Đang chạy suy luận public_test v8.0 với {PRIMARY_LORA} -> {SUBMISSION_V8}")
!python3 -X utf8 -u scripts/infer.py --data-root "{DATA_ROOT}" --split public_test \
  --asr-mode transformers --asr-model "{TURBO_OUT}" --asr-secondary "{PHO_OUT}" \
  --asr-cache "{ASR_CACHE_FT}" {base_cache_arg} \
  --llm-backend transformers --llm-model "{PRIMARY_LORA}" --dtype {DTYPE} \
  --time-budget {TIME_BUDGET} --out "{SUBMISSION_V8}" --timing-out "{TIMING_V8}" --resume


In [ ]:
# === Ô 7: Kiểm tra hợp lệ Schema zaic_tools · Thống kê Timing & So sánh v5 / v6 / v7 / v8 ===
import os, json
os.chdir(WORK_DIR)
SUBMISSION_V8 = f"{OUT_DIR}/submission_v8.jsonl"
TIMING_V8     = f"{OUT_DIR}/timing_public_v8.jsonl"

for tag, p in [
    ("v5", get_path("submission_v5.jsonl")),
    ("v6", get_path("submission_v6.jsonl")),
    ("v7", get_path("submission_v7.jsonl")),
    ("v8", SUBMISSION_V8),
]:
    if os.path.isfile(p):
        rows = [json.loads(line) for line in open(p, encoding="utf-8") if line.strip()]
        empty_n = sum(1 for r in rows if not r.get("tool_calls"))
        total_c = sum(len(r.get("tool_calls", [])) for r in rows)
        avg_c = total_c / max(1, len(rows))
        pct_e = 100.0 * empty_n / max(1, len(rows))
        print(f"📊 [{tag}] Số dòng: {len(rows)}/634 | Tổng calls: {total_c} ({avg_c:.2f} calls/mẫu) | Nộp [] : {empty_n} ({pct_e:.1f}%)")

!PYTHONPATH="{DATA_ROOT}/public_test" python3 -X utf8 -u -m zaic_tools --split public_test --root "{DATA_ROOT}" --validate "{SUBMISSION_V8}"

if os.path.isfile(TIMING_V8):
    sec = sorted(json.loads(l)["seconds"] for l in open(TIMING_V8, encoding="utf-8") if l.strip())
    n = len(sec)
    print(f"⏱ Thời gian / mẫu (v8): TB {sum(sec)/n:.2f}s | p50 {sec[n//2]:.2f}s | p95 {sec[int(.95*n)]:.2f}s | max {sec[-1]:.2f}s")
print(f"🎉 HOÀN TẤT 100%! File nộp bài đã đồng bộ trên Google Drive tại: {SUBMISSION_V8}")


In [ ]:
# === Ô 8 (Tùy chọn khi đóng gói Docker nộp BTC cho đội AIO_HacDaiBangBang): Merge PureLoRA v8 vào Qwen3-4B & Gom saved_models/ ===
# Chỉ chạy ô này (đổi RUN_DOCKER_MODEL_EXPORT = True) khi bạn muốn xuất thư mục saved_models/ (asr_turbo_ft_v1, asr_pho_ft_v1, qwen3_4b_sft_merged)
RUN_DOCKER_MODEL_EXPORT = False
if RUN_DOCKER_MODEL_EXPORT:
    import os
    os.chdir(WORK_DIR)
    SAVED_MODELS_DIR = f"{OUT_DIR}/saved_models"
    LORA_V8_OUT = f"{OUT_DIR}/qwen3_4b_sft_lora_v8"
    !python3 -X utf8 -u scripts/prepare_docker_models.py \
      --out-dir "{SAVED_MODELS_DIR}" \
      --asr-turbo "{TURBO_OUT}" \
      --asr-pho "{PHO_OUT}" \
      --lora-dir "{LORA_V8_OUT}"
